# Layer 2: fully nested leave-one-farm-out evaluation

In [ ]:
from __future__ import annotations

import sys
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

for _cand in (Path.cwd() / "notebooks", Path.cwd(), Path.cwd().parent):
    if (_cand / "crossfarm_nested.py").exists():
        sys.path.insert(0, str(_cand))
        break
import crossfarm_nested as cn

OUT_DIR = cn.OUT / "14n_layer2_crossfarm_nested"
LOG_DIR = cn.OUT / "logs"
for _d in (OUT_DIR, LOG_DIR):
    _d.mkdir(parents=True, exist_ok=True)
RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
LOG_PATH = LOG_DIR / f"13n_layer2_crossfarm_fully_nested_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}.log"


def log(msg):
    line = f"[{datetime.now(timezone.utc):%Y-%m-%d %H:%M:%S UTC}] {msg}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")


clips = cn.load_clip_table()
segs = {name: cn.load_segmentation(name) for name in cn.SEGMENTATIONS}
log(f"Loaded {len(clips)} clips and segmentations {list(segs)}.")

In [ ]:
log("Full corpus:")
full = cn.choose_layer1_settings(set(clips["clip_id"]), clips, segs, log=log)
full["scorecard"].assign(scope="full_corpus").to_csv(OUT_DIR / "layer1_scorecard_full_corpus.csv", index=False)
full["kstudy"].assign(scope="full_corpus").to_csv(OUT_DIR / "layer1_kstudy_full_corpus.csv", index=False)

In [ ]:
known = clips[clips["farm_num"].isin(cn.FARMS)]
fold_settings, cards, kstudies, cand_k = {}, [], [], []
for held in cn.FARMS:
    train_set = set(known.loc[known["farm_num"] != held, "clip_id"])
    log(f"Held-out farm {held}: {len(train_set)} training clips")
    res = cn.choose_layer1_settings(train_set, clips, segs, log=log)
    fold_settings[held] = {"segmentation": res["segmentation"], "k": res["k"]}
    cards.append(res["scorecard"].assign(heldout_farm=held))
    kstudies.append(res["kstudy"].assign(heldout_farm=held))
    cand_k.append(res["candidate_kstudies"].assign(heldout_farm=held))
pd.concat(cards).to_csv(OUT_DIR / "layer1_scorecard_by_fold.csv", index=False)
pd.concat(kstudies).to_csv(OUT_DIR / "layer1_kstudy_by_fold.csv", index=False)
pd.concat(cand_k).to_csv(OUT_DIR / "layer1_candidate_kstudy_by_fold.csv", index=False)
fold_settings

In [ ]:
pred_df, selection_df, fold_info = cn.run_lofo(clips, fold_settings, nested_inner=True, log=log)
metrics_df, perfarm_df = cn.summarise(pred_df)

ablation_metrics, ablation_perfarm, ablation_folds, ablation_selected = [], [], [], []
for variant in cn.VOCAB_VARIANTS:
    p_v, _, f_v = cn.run_lofo(clips, fold_settings, nested_inner=True, vocab_variants=[variant], log=log)
    m_v, pf_v = cn.summarise(p_v)
    ablation_metrics.append(m_v.assign(vocabulary=variant))
    ablation_perfarm.append(pf_v.assign(vocabulary=variant))
    ablation_folds.append(f_v.assign(vocabulary=variant))
    sel = p_v.groupby(["heldout_farm", "target_name"])["bundle_name"].first().reset_index()
    ablation_selected.append(sel.assign(vocabulary=variant))

fixed = {h: {"segmentation": full["segmentation"], "k": full["k"]} for h in cn.FARMS}
sensitivity = []
for label, nested in (("fixed_C_K_full_corpus__nested_inner", True),
                      ("fixed_C_K_full_corpus__non_nested_inner", False)):
    p_s, _, _ = cn.run_lofo(clips, fixed, nested_inner=nested, log=log)
    m_s, _ = cn.summarise(p_s)
    sensitivity.append(m_s.assign(setting=label))
sensitivity.append(metrics_df.assign(setting="per_fold_choice__nested_inner"))

In [ ]:
pred_df.to_csv(OUT_DIR / "crossfarm_fold_predictions.csv.gz", index=False, compression="gzip")
selection_df.to_csv(OUT_DIR / "crossfarm_bundle_selection_scores.csv", index=False)
metrics_df.to_csv(OUT_DIR / "crossfarm_metrics_summary.csv", index=False)
perfarm_df.to_csv(OUT_DIR / "crossfarm_bestbundle_perfarm_metrics.csv", index=False)
fold_info.to_csv(OUT_DIR / "crossfarm_fold_settings.csv", index=False)
pd.concat(ablation_metrics).to_csv(OUT_DIR / "vocabulary_ablation_metrics.csv", index=False)
pd.concat(ablation_perfarm).to_csv(OUT_DIR / "vocabulary_ablation_perfarm.csv", index=False)
pd.concat(ablation_folds).to_csv(OUT_DIR / "vocabulary_ablation_fold_settings.csv", index=False)
pd.concat(ablation_selected).to_csv(OUT_DIR / "vocabulary_ablation_selected_bundles.csv", index=False)
sensitivity_df = pd.concat(sensitivity)
sensitivity_df.to_csv(OUT_DIR / "layer1_sensitivity_metrics.csv", index=False)

with open(OUT_DIR / "crossfarm_manifest.json", "w", encoding="utf-8") as f:
    json.dump({"build_ts_utc": RUN_TS,
               "fold_settings": {str(k): v for k, v in fold_settings.items()},
               "full_corpus_settings": {"segmentation": full["segmentation"], "k": full["k"]},
               "k_values": cn.K_VALUES, "kstudy_seeds": cn.KSTUDY_SEEDS,
               "bias_audit_fields": cn.LOFO_AUDIT_FIELDS,
               "inner_loop": "vocabulary and features rebuilt from the inner training farm"}, f, indent=2)
log("Saved outputs.")
lr = lambda d: d[d.model_name == "logreg_balanced"]
pd.concat([lr(d)[["target_name", "macro_f1"]].assign(run=d["setting"].iloc[0]) for d in sensitivity]
          + [lr(d)[["target_name", "macro_f1"]].assign(run="vocabulary_" + d["vocabulary"].iloc[0])
             for d in ablation_metrics]).pivot(index="target_name", columns="run", values="macro_f1").round(3)

In [ ]:
display(fold_info)
display(perfarm_df)